In [1]:
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/data_utils.py

In [2]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.feature_extraction import text as sklearn_text

import numpy as np
import pandas as pd
import spacy
import torch.nn.functional as F
import torch
import umap
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import textwrap
from tqdm.notebook import tqdm
import re
import json

from google.colab import files
from data_utils import balance_score, distance_score, silhouette_score, display_silhouette_plots

# Load data and initialize model

In [3]:
# load data
artists_df = pd.read_csv('./artists.csv')
exhibitions_df = pd.read_csv('./exhibitions.csv')

artist_texts_df = pd.read_csv('./all_artist_texts.csv')
institution_texts_df = pd.read_csv('./all_institution_texts.csv')

artworks_df = pd.read_csv('./artworks.csv')
institutions_df = pd.read_csv('./institutions.csv')

# rename unnamed index
artist_texts_df.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
institution_texts_df.rename(columns={'Unnamed: 0': 'id'}, inplace=True)

institution_texts_df.head()

,id,institution,text,source_idx,artist,is_artwork
0,0,0,"American, born 1975",IT01,1,False
1,1,0,American,IT02,1,False
2,2,0,female,IT03,1,False
3,3,0,"American, born 1975",IT04,2,False
4,4,0,American,IT05,2,False


In [4]:
nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

if torch.cuda.is_available():
    print("using GPU")
    model.to('cuda')
else:
    print("using CPU")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

using GPU


# Text parsing helpers

*   semantic_chunk
*   get_word_weights
*   get_chunks_for_df


In [ ]:
# based on semantic similarity between sentences
def semantic_chunk(text, threshold=0.75, min_sentences=1, min_chars=100, max_sentences=3, max_chunk_chars=300):
    # replace unended double new lines for splitting equally
    processed_text = re.sub(r'\n\n+', '. ', text)
    doc = nlp(processed_text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    if len(sentences) <= min_sentences:
        return [text]

    # embed sentences to compare similarity
    embeddings = model.encode(
        [f"search_document: {s}" for s in sentences],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        # compare current sentence to mean of the current chunk
        chunk_embedding = embeddings[i - len(current_chunk):i].mean(axis=0, keepdims=True)
        sim = cosine_similarity(chunk_embedding, embeddings[i:i+1])[0][0]

        current_char_count = sum(len(s) for s in current_chunk)

        meets_min_sentences = len(current_chunk) >= min_sentences
        meets_min_chars = current_char_count >= min_chars
        semantically_different = sim < threshold
        too_long_sentences = len(current_chunk) >= max_sentences
        too_long_chars = current_char_count >= max_chunk_chars

        force_split = too_long_sentences or too_long_chars

        conditional_semantic_split = semantically_different and meets_min_sentences and meets_min_chars

        should_split = force_split or conditional_semantic_split

        if should_split:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
        else:
            current_chunk.append(sentences[i])

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [6]:
def get_word_weights(doc, vectorizer):
  doc_tfidf = vectorizer.transform([doc])
  feature_names = vectorizer.get_feature_names_out()

  df = pd.DataFrame(doc_tfidf[0].T.todense(),
                    index=feature_names,
                    columns=["tfidf"])

  ret = df.sort_values(by="tfidf", ascending=False)

  return ret.head(3).index.tolist()


In [7]:
def get_chunks_for_df(df, process_artworks, min_sentences=1):
  chunks_arr = []

  # Add tqdm to the iterrows loop for a progress bar
  for idx, row in tqdm(df.iterrows(), total=len(df), desc="Chunking texts"):
    if (row["text"] is not None):
      if (row["is_artwork"] and not process_artworks):
        # skip processing for artwork titles
        chunks_arr.append({
            "source_idx": row["source_idx"],
            "text": row["text"],
            "is_artwork": row["is_artwork"]
        })
      else:
        chunks = semantic_chunk(row["text"], 0.75, min_sentences)
        for chunk in chunks:
            chunks_arr.append({
                "source_idx": row["source_idx"],
                "text": chunk,
                "is_artwork": row["is_artwork"]
            })


  chunks_df = pd.DataFrame(chunks_arr)
  return chunks_df

In [28]:
def semantic_chunk(text, threshold=0.75, min_sentences=1, min_chars=100, max_sentences=3, max_chunk_chars=300):
    # replace double new lines with an explicit marker  to force a chunk split.
    processed_text = re.sub(r'\n\n+', '\n<PARAGRAPH_SPLIT_MARKER>\n', text)
    doc = nlp(processed_text)

    sentences_raw = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    if not sentences_raw:
        return []

    sentences_to_embed_flat = []
    original_idx_to_flat_indices = []

    current_flat_idx = 0
    for s_raw_text in sentences_raw:
        if '<PARAGRAPH_SPLIT_MARKER>' in s_raw_text:
            # if the sentence contains the marker, split it to extract actual text parts and discard the text
            parts = [p.strip() for p in s_raw_text.split('<PARAGRAPH_SPLIT_MARKER>') if p.strip()]
            if parts:
                original_idx_to_flat_indices.append((current_flat_idx, len(parts)))
                sentences_to_embed_flat.extend(parts)
                current_flat_idx += len(parts)
            else:
                original_idx_to_flat_indices.append((-1, 0))
        else:
            # add clean sentences to be embedded
            original_idx_to_flat_indices.append((current_flat_idx, 1))
            sentences_to_embed_flat.append(s_raw_text)
            current_flat_idx += 1

    if not sentences_to_embed_flat:
        return []

    embeddings_flat = model.encode(
        [f"search_document: {s}" for s in sentences_to_embed_flat],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    chunks = []
    current_chunk_sentences = []
    current_chunk_embeddings = []

    for i, sentence_raw_text in enumerate(sentences_raw):
        start_flat_idx, num_flat_sentences = original_idx_to_flat_indices[i]

        # for forced split points with no actual text
        if start_flat_idx == -1:
            if current_chunk_sentences:
                chunks.append(" ".join(current_chunk_sentences))
                current_chunk_sentences = []
                current_chunk_embeddings = []
            continue

        # iterate over extracted clean sentences
        for j in range(num_flat_sentences):
            sentence_to_process = sentences_to_embed_flat[start_flat_idx + j]
            embedding_to_process = embeddings_flat[start_flat_idx + j]

            prospective_chunk_len_sentences = len(current_chunk_sentences) + 1
            prospective_chunk_len_chars = sum(len(s) for s in current_chunk_sentences) + len(sentence_to_process)

            force_max_split = prospective_chunk_len_sentences > max_sentences or prospective_chunk_len_chars > max_chunk_chars

            semantically_different_split = False
            if current_chunk_sentences:
                current_char_count = sum(len(s) for s in current_chunk_sentences)
                meets_min_sentences_for_sem = len(current_chunk_sentences) >= min_sentences
                meets_min_chars_for_sem = current_char_count >= min_chars

                chunk_embedding = np.mean(current_chunk_embeddings, axis=0, keepdims=True)
                sim = cosine_similarity(chunk_embedding, embedding_to_process.reshape(1, -1))[0][0]
                semantically_different_split = sim < threshold and meets_min_sentences_for_sem and meets_min_chars_for_sem

            should_split = False
            if current_chunk_sentences and (force_max_split or semantically_different_split):
                should_split = True

            if should_split:
                chunks.append(" ".join(current_chunk_sentences))
                current_chunk_sentences = [sentence_to_process]
                current_chunk_embeddings = [embedding_to_process]
            else:
                current_chunk_sentences.append(sentence_to_process)
                current_chunk_embeddings.append(embedding_to_process)

    # add the current sentences if not empty
    if current_chunk_sentences:
        chunks.append(" ".join(current_chunk_sentences))

    return chunks

# Prep data

In [29]:
# get all exhibition text
exhibitions_df['full_text'] = exhibitions_df['name'].fillna('') + ". " + exhibitions_df['description'].fillna('')

In [54]:
# chunk artist texts
ar_texts = get_chunks_for_df(artist_texts_df, False, 2)

Chunking texts:   0%|          | 0/584 [00:00<?, ?it/s]

In [44]:
print(len(ar_texts))
ar_texts.head()

1791


,source_idx,text,is_artwork,identity_relevance
0,AT01,I wanted to directly challenge previous assert...,False,0.575999
1,AT01,I’ve always been interested in the shared real...,False,0.555902
2,AT01,Locating undiscovered territory or dimensional...,False,0.597699
3,AT01,The important thing is that I wasn’t intereste...,False,0.586456
4,AT01,The image that results is both mannered and im...,False,0.555297


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [32]:
# chunk inst texts
inst_texts = get_chunks_for_df(institution_texts_df, True, 1)

Chunking texts:   0%|          | 0/338 [00:00<?, ?it/s]

In [33]:
print(len(inst_texts))
inst_texts.head()

846


,source_idx,text,is_artwork
0,IT01,"American, born 1975",False
1,IT02,American,False
2,IT03,female,False
3,IT04,"American, born 1975",False
4,IT05,American,False


# Labeling - stop words

In [34]:
new_stop_words = ["biennale", "biennial", "museum", "exhibition", "baratta", "italian", "year", "s", "art", "artist", "th", "arsenale", "tam"]
custom_stop_words = list(sklearn_text.ENGLISH_STOP_WORDS.union(new_stop_words))

In [35]:
# add all the artist first and last names to the stop words list
for idx, row in artists_df.iterrows():
    # split by first and last name
    names = re.split(r'[- ]', row["name"])

    for name in names:
      custom_stop_words.append(name.lower())

# add all locations
for idx, row in exhibitions_df.iterrows():
  locations = re.split(r'[, ]', row["location"])

  for location in locations:
    custom_stop_words.append(location.lower())

In [36]:
def remove_numbers(text):
    return re.sub(r'\d+', '', text.lower())

# Exhibition labels

In [37]:
all_corpus = exhibitions_df['full_text'].tolist()

vectorizer = TfidfVectorizer(preprocessor=remove_numbers, stop_words=custom_stop_words)

tfidf_matrix = vectorizer.fit_transform(all_corpus)

In [38]:
# get word weights for each row in exhibitions
exhibitions_df["labels"] = exhibitions_df.apply(lambda row: get_word_weights(row['full_text'], vectorizer), axis=1)
exhibitions_df["labels"]

,labels
0,"[hapa, ancestry, asian]"
1,"[mixed, asian, american]"
2,"[citizen, universe, life]"
3,"[orient, dis, diaspora]"
4,"[legacies, asian, american]"
5,"[east, asian, chinese]"
6,"[countries, international, world]"
7,"[ideologies, global, authoritarian]"
8,"[korean, english, pacific]"
9,"[hard, american, america]"


In [39]:
# download w labels
exhibitions_df.to_csv('exhibitions_with_labels.csv', index=False)

files.download('exhibitions_with_labels.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Filter for identity-specific texts

For artists

In [52]:
query_words = [
    # original
    "identity", "persona", "race", "ethnicity", "culture",
    "self", "category", "belonging", "community",
    "ancestry", "generation", "experience", "in-between",

    # mixed race
    "mixed", "multiracial", "biracial", "half", "hyphenated",
    "passing", "blood", "skin", "contradict"

    # diaspora
    "diaspora", "diasporic", "transnational", "immigrant", "migration",
    "displacement", "exile", "homeland", "motherland", "origin",

    # in betweenness
    "liminal", "borderlands", "threshold", "assimilation", "integration",

    # heritage
    "heritage", "roots", "memory", "nostalgia", "home", "tradition",
    "language", "birth", "death", "born", "nationality", "died"

    # visibility
    "otherness", "outsider", "minority", "visibility", "authenticity",
    "representation", "erasure", "recognition", "voice", "truth",
    "invisible", "seen", "silence", "absence"
]

In [61]:
query = "search_query: " + " ".join(query_words)

query_embedding = model.encode(query, convert_to_numpy=True, normalize_embeddings=True)

chunk_embeddings_for_filter = model.encode(
    ["search_document: " + t for t in ar_texts["text"]],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

sims = cosine_similarity([query_embedding], chunk_embeddings_for_filter)[0]
ar_texts["identity_relevance"] = sims

# keep all artworks, discard texts that are not artworks
# whose identity relevance is below the threshold
threshold = 0.58
filtered_ar_texts = ar_texts[
    (ar_texts["is_artwork"] == True) |
    ((ar_texts["is_artwork"] == False) & (ar_texts["identity_relevance"] >= threshold))
].reset_index(drop=True)

print(f"Rows remaining: {len(filtered_ar_texts)}")

Batches:   0%|          | 0/237 [00:00<?, ?it/s]

Rows remaining: 952


In [59]:
texts_only = filtered_ar_texts[filtered_ar_texts["is_artwork"] == False]
texts_only.to_csv('ar_texts_filtered.csv', index=False)
files.download('ar_texts_filtered.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Embed all texts

For artists

In [62]:
ar_texts = filtered_ar_texts

In [63]:
ar_texts_arr = ["clustering: " + t for t in ar_texts["text"].tolist()]

# embed
ar_embeddings = model.encode(
    ar_texts_arr,
    batch_size=64,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

ar_embeddings_np = ar_embeddings.cpu().numpy()

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

For institutions

In [64]:
inst_texts_arr = ["clustering: " + t for t in inst_texts["text"].tolist()]

# embed
inst_embeddings = model.encode(
    inst_texts_arr,
    batch_size=64,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

inst_embeddings_np = inst_embeddings.cpu().numpy()

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

# Build lookups

In [65]:
artist_texts_source_idx_map = artist_texts_df.set_index('source_idx')
institution_texts_source_idx_map = institution_texts_df.set_index('source_idx')

artworks_source_idx_map = artworks_df.set_index('id')

In [66]:
def map_metadata(row, mode):
    artist_id = np.nan
    institution_id = np.nan

    if mode == 'artist':
      source_idx_map = artist_texts_source_idx_map
    else:
      source_idx_map = institution_texts_source_idx_map

    if row['is_artwork']:
        artwork_row = artworks_source_idx_map.loc[row['source_idx']]
        artist_id = artwork_row['artist']
        institution_id = artwork_row['institution']
    else:
        artist_id = source_idx_map.loc[row['source_idx'], 'artist']
        if mode != 'artist':
          institution_id = source_idx_map.loc[row['source_idx'], 'institution']

    return pd.Series({'artist_id': artist_id, 'institution_id': institution_id})

In [67]:
artist_id_to_name = artists_df.set_index("id")["name"].to_dict()
inst_id_to_name = institutions_df.set_index("id")["name"].to_dict()

ar_lookup_df = ar_texts[["text", "source_idx", "is_artwork"]].copy()

ar_lookup_df[['artist_id', 'institution_id']] = ar_lookup_df.apply(lambda x: map_metadata(x, 'artist'), axis=1)

ar_lookup_df['artist_id'] = ar_lookup_df['artist_id'].astype(pd.Int64Dtype())
ar_lookup_df['institution_id'] = ar_lookup_df['institution_id'].astype(pd.Int64Dtype())

# for testing, map to names to see accuracy
ar_lookup_df['artist_name'] = ar_lookup_df['artist_id'].map(artist_id_to_name)
ar_lookup_df['institution_name'] = ar_lookup_df['institution_id'].map(inst_id_to_name)

ar_lookup_df

,text,source_idx,is_artwork,artist_id,institution_id,artist_name,institution_name
0,Locating undiscovered territory or dimensional...,AT01,False,1,<NA>,Amanda Ross-Ho,NaN
1,The important thing is that I wasn’t intereste...,AT01,False,1,<NA>,Amanda Ross-Ho,NaN
2,My practice originates in the act of negotiati...,AT04,False,1,<NA>,Amanda Ross-Ho,NaN
3,I weave together data from a broad hierarchy o...,AT04,False,1,<NA>,Amanda Ross-Ho,NaN
4,"You already mentioned Lolita, which deals with...",AT09,False,2,<NA>,Laurel Nakadate,NaN
...,...,...,...,...,...,...,...
947,Belly Painting: Prussian Blue,W405,True,12,7,Byron Kim,National Gallery of Art
948,Untitled,W406,True,29,7,Hung Liu,National Gallery of Art
949,Post-Age,W407,True,29,7,Hung Liu,National Gallery of Art
950,Seoul Home – Inverted,W408,True,6,7,Do Ho Suh,National Gallery of Art


In [68]:
inst_lookup_df = inst_texts[["text", "source_idx", "is_artwork"]].copy()

inst_lookup_df[['artist_id', 'institution_id']] = inst_lookup_df.apply(lambda x: map_metadata(x, 'institution'), axis=1)

inst_lookup_df['artist_id'] = inst_lookup_df['artist_id'].astype(pd.Int64Dtype())
# inst_lookup_df['institution_id'] = inst_lookup_df['institution_id'].astype(pd.Int64Dtype())

# add names for testing
inst_lookup_df['artist_name'] = inst_lookup_df['artist_id'].map(artist_id_to_name)
inst_lookup_df['institution_name'] = inst_lookup_df['institution_id'].map(inst_id_to_name)

inst_lookup_df

,text,source_idx,is_artwork,artist_id,institution_id,artist_name,institution_name
0,"American, born 1975",IT01,False,1,0,Amanda Ross-Ho,Museum of Modern Art
1,American,IT02,False,1,0,Amanda Ross-Ho,Museum of Modern Art
2,female,IT03,False,1,0,Amanda Ross-Ho,Museum of Modern Art
3,"American, born 1975",IT04,False,2,0,Laurel Nakadate,Museum of Modern Art
4,American,IT05,False,2,0,Laurel Nakadate,Museum of Modern Art
...,...,...,...,...,...,...,...
841,The facial features are partially visible in p...,W404,True,21,7,Alfonso Ossorio,National Gallery of Art
842,They are not shown wearing any jewelry or hold...,W404,True,21,7,Alfonso Ossorio,National Gallery of Art
843,The image shows a composition of lines and sha...,W406,True,29,7,Hung Liu,National Gallery of Art
844,"In the left oval, there is a pattern resemblin...",W406,True,29,7,Hung Liu,National Gallery of Art


In [69]:
inst_lookup_df

,text,source_idx,is_artwork,artist_id,institution_id,artist_name,institution_name
0,"American, born 1975",IT01,False,1,0,Amanda Ross-Ho,Museum of Modern Art
1,American,IT02,False,1,0,Amanda Ross-Ho,Museum of Modern Art
2,female,IT03,False,1,0,Amanda Ross-Ho,Museum of Modern Art
3,"American, born 1975",IT04,False,2,0,Laurel Nakadate,Museum of Modern Art
4,American,IT05,False,2,0,Laurel Nakadate,Museum of Modern Art
...,...,...,...,...,...,...,...
841,The facial features are partially visible in p...,W404,True,21,7,Alfonso Ossorio,National Gallery of Art
842,They are not shown wearing any jewelry or hold...,W404,True,21,7,Alfonso Ossorio,National Gallery of Art
843,The image shows a composition of lines and sha...,W406,True,29,7,Hung Liu,National Gallery of Art
844,"In the left oval, there is a pattern resemblin...",W406,True,29,7,Hung Liu,National Gallery of Art


# Helpers to cluster for n

In [71]:
def run_clustering_and_evaluate(embeddings, n_clusters):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
    cluster_labels = kmeans.fit_predict(embeddings)

    sc = kmeans.score(embeddings)
    distance_sc = distance_score(embeddings, cluster_labels)
    silhouette_sc = silhouette_score(embeddings, cluster_labels)
    balance_sc = balance_score(cluster_labels)

    print(f"\n{'='*80}")
    print(f"FOR NUM CLUSTERS {n_clusters} --------------------")
    print("score: ", sc)
    print("distance score: ", distance_sc)
    print("silhouette score: ", silhouette_sc)
    print("balance score: ", balance_sc)

    return kmeans, cluster_labels

In [72]:
def calculate_distances(embeddings_np, kmeans):
    centers = kmeans.cluster_centers_
    all_distances = euclidean_distances(embeddings_np, centers)

    return all_distances

In [73]:
def extract_cluster_keywords(df_source, cluster_labels, n_clusters, stop_words):
    temp_df = df_source.copy()
    temp_df["cluster"] = cluster_labels

    cluster_texts = []
    for c in range(n_clusters):
        mask = temp_df["cluster"] == c
        combined = " ".join(temp_df.loc[mask, "text"].tolist())
        cluster_texts.append(combined)

    vectorizer = TfidfVectorizer(preprocessor=remove_numbers, stop_words=stop_words, token_pattern=r'\b\w*[a-zA-Z]\w*\b', max_features=1000, ngram_range=(1,2))
    tfidf_matrix = vectorizer.fit_transform(cluster_texts)
    feature_names = vectorizer.get_feature_names_out()

    top_n = 5
    cluster_keywords = {}
    for c in range(n_clusters):
        top_indices = np.argsort(tfidf_matrix[c].toarray()[0])[::-1][:top_n]
        keywords = [feature_names[i] for i in top_indices]
        cluster_keywords[str(c)] = keywords

    return cluster_keywords

In [74]:
def get_cluster_results_df(df_source, cluster_labels, all_distances, n_clusters):
    own_center_distances = all_distances[range(len(cluster_labels)), cluster_labels]

    masked = all_distances.copy()
    masked[range(len(cluster_labels)), cluster_labels] = np.inf
    closest_other_cluster = np.argmin(masked, axis=1)
    closest_other_cluster_distance = masked[range(len(cluster_labels)), closest_other_cluster]

    results_df = pd.DataFrame({
        "point":                 range(len(cluster_labels)),
        "cluster":               cluster_labels,
        "distance_to_center":    own_center_distances,
        "text":            df_source["text"].values,
        "artist_id":        df_source["artist_id"],
        "is_artwork":      df_source["is_artwork"],
        "source_idx":            df_source["source_idx"].values
    })

    for c in range(n_clusters):
      results_df[f"dist_to_cluster_{c}"] = all_distances[:, c]

    if 'institution_id' in df_source.columns:
        results_df["institution_id"] = df_source["institution_id"].values

    if 'artist_name' in df_source.columns:
        results_df["artist_name"] = df_source["artist_name"].values

    return results_df

In [75]:
def get_all_clusters(fname, embeddings, lookups, n_clusters):
  all_json_data = []

  for k in range(12, n_clusters):
      kmeans_model, labels = run_clustering_and_evaluate(embeddings, k)

      distances = calculate_distances(embeddings, kmeans_model)

      keywords_dict = extract_cluster_keywords(lookups, labels, k, custom_stop_words)

      results_df = get_cluster_results_df(lookups, labels, distances, k)

      cluster_list = []
      for c in range(k):
          c_str = str(c)
          cluster_data = results_df[results_df['cluster'] == c].to_dict(orient='records')
          cluster_list.append({
              "key": c_str,
              "keywords": keywords_dict.get(c_str, []),
              "data": cluster_data
          })

      all_json_data.append({
          "n": k,
          "clusters": cluster_list
      })

  # save to single json
  filename = fname + '.json'
  with open(filename, 'w', encoding='utf-8') as f:
      json.dump(all_json_data, f, indent=4, ensure_ascii=False)

  files.download(filename)

# Cluster by n

In [76]:
get_all_clusters('inst_clusters', inst_embeddings_np, inst_lookup_df, 31)


FOR NUM CLUSTERS 12 --------------------
score:  -218.58262634277344
distance score:  0.5016561
silhouette score:  0.060894445
balance score:  0.8201160541586074

FOR NUM CLUSTERS 13 --------------------
score:  -218.40841674804688
distance score:  0.50141263
silhouette score:  0.056782763
balance score:  0.8096926713947991


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(



FOR NUM CLUSTERS 14 --------------------
score:  -216.47799682617188
distance score:  0.4990519
silhouette score:  0.058424808
balance score:  0.784506273867976

FOR NUM CLUSTERS 15 --------------------
score:  -211.7406463623047
distance score:  0.49400604
silhouette score:  0.06647594
balance score:  0.82193515704154


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(



FOR NUM CLUSTERS 16 --------------------
score:  -209.13314819335938
distance score:  0.49237537
silhouette score:  0.06831372
balance score:  0.8490149724192277


...

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(



FOR NUM CLUSTERS 30 --------------------
score:  -188.26499938964844
distance score:  0.44598013
silhouette score:  0.07396973
balance score:  0.7794081682562973


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [77]:
get_all_clusters('ar_clusters', ar_embeddings_np, ar_lookup_df, 31)


FOR NUM CLUSTERS 12 --------------------
score:  -255.0402374267578
distance score:  0.42892265
silhouette score:  0.06041533
balance score:  0.6768525592055004

FOR NUM CLUSTERS 13 --------------------
score:  -252.41384887695312
distance score:  0.43199173
silhouette score:  0.061507747
balance score:  0.7125350140056023


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(



FOR NUM CLUSTERS 14 --------------------
score:  -250.17845153808594
distance score:  0.4367725
silhouette score:  0.06459873
balance score:  0.7126696832579185

FOR NUM CLUSTERS 15 --------------------
score:  -248.4674835205078
distance score:  0.44138667
silhouette score:  0.068235055
balance score:  0.7009303721488596


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(



FOR NUM CLUSTERS 16 --------------------
score:  -247.29110717773438
distance score:  0.4459896
silhouette score:  0.067733765
balance score:  0.7389355742296919

FOR NUM CLUSTERS 17 --------------------
score:  -246.4243927001953
distance score:  0.44981727
silhouette score:  0.06788917
balance score:  0.7377232142857143


...

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(



FOR NUM CLUSTERS 30 --------------------
score:  -228.3011932373047
distance score:  0.41693977
silhouette score:  0.08006394
balance score:  0.7373225152129818


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['c', 'd'] not in stop_words.
  warnings.warn(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Combine artist-specific texts from both sources

In [78]:
combined_artist_data = []

ar_data_for_combine = ar_lookup_df.copy()
ar_data_for_combine = ar_data_for_combine[['artist_id', 'institution_id', 'is_artwork', 'source_idx', 'text']]

inst_data_for_combine = inst_lookup_df.copy()
inst_data_for_combine = inst_data_for_combine[['artist_id', 'institution_id', 'is_artwork', 'source_idx', 'text']]

all_artist_points_df = pd.concat([ar_data_for_combine, inst_data_for_combine], ignore_index=True)

display(all_artist_points_df.head())

all_artist_points_df.to_csv('all_artist_combined_texts.csv')
files.download('all_artist_combined_texts.csv')

,artist_id,institution_id,is_artwork,source_idx,text
0,1,<NA>,False,AT01,Locating undiscovered territory or dimensional...
1,1,<NA>,False,AT01,The important thing is that I wasn’t intereste...
2,1,<NA>,False,AT04,My practice originates in the act of negotiati...
3,1,<NA>,False,AT04,I weave together data from a broad hierarchy o...
4,2,<NA>,False,AT09,"You already mentioned Lolita, which deals with..."


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>